In [4]:
!pip install google-genai langchain faiss-cpu python-dotenv
!pip install langchain[all]
!pip install sentence-transformers
!pip install datasets huggingface-hub



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import requests as rq
import bs4 
import json
from dotenv import load_dotenv
import langchain as lc
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI


In [6]:
import requests as rq
from bs4 import BeautifulSoup
import json

def _search_google(search):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }
    resp = rq.get(
        "https://www.google.com/search",
        params={"q": search, "hl": "en"},
        headers=headers,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []
    for item in soup.select("div.g")[:5]:
        title_tag = item.select_one("h3")
        link_tag = item.select_one("a")
        snippet_tag = item.select_one(".VwiC3b, .IsZvec")
        if title_tag and link_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": link_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def _search_duckduckgo(search):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }
    resp = rq.get(
        "https://html.duckduckgo.com/html",
        params={"q": search},
        headers=headers,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []
    for item in soup.select(".result"):
        title_tag = item.select_one("a.result__a")
        snippet_tag = item.select_one("a.result__snippet, .result__snippet")
        if title_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": title_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def _search_bing(search):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }
    resp = rq.get(
        "https://www.bing.com/search",
        params={"q": search},
        headers=headers,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []
    for item in soup.select("li.b_algo"):
        title_tag = item.select_one("h2 a")
        snippet_tag = item.select_one("p")
        if title_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": title_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def search_engine(search):
    for search_fn in (_search_google, _search_duckduckgo, _search_bing):
        results = search_fn(search)
        if results:
            return json.dumps({"search": search, "results": results}, ensure_ascii=False)

    return json.dumps({
        "search": search,
        "results": [],
        "error": "No results returned by Google, DuckDuckGo, or Bing.",
    }, ensure_ascii=False)

print(search_engine("nice"))


{"search": "nice", "results": [{"title": "Nice - Klook-Up To 60% Off Activities", "link": "//duckduckgo.com/l/?uddg=https%3A%2F%2Fduckduckgo.com%2Fy.js%3Fad_domain%3Dklook.com%26ad_provider%3Dbingv7aa%26ad_type%3Dtxad%26click_metadata%3DTBkHFd0JoP6PZZQiaB8wFi4JsSJ%252DasKkzvzH265E3PvvTUHEMFN8Txk%252DPNnK9hYTvyWtB1AmGL4mmyQ_7km_yPwu%252DIJl2ZuYtz3zJ22BnVIJDfomAPQs7NXb8J8Bhh6ratEtGMYIMfN55ezk91TviQyQK%252DA4nAzdBTNSrWTsnxI.FTcBFfh4iUzyaRHlKe82eA%26rut%3Db294954aaf8d55c218634b02b26c4baa7783dad39df31a4aca78b9214efa07b2%26u3%3Dhttps%253A%252F%252Fwww.bing.com%252Faclick%253Fld%253De87Fhfhdx2xQJ3DrJstrJ_rjVUCUx7vP09CtqbOWryRQrwZQL6YXwirhy6BpB%252D%252DxiiIatC9pf_kZnzb2kmVusH19kE2I9x3uvHwOz4u32YIeGRiIofYsV59JdevhmpdDXmYOG0AQqgyecRFJm8BV1zMOPkLwiNBt_w%252DlifsAKNMl_hiYobT6xDDeSfqpZnYTdN058nqBVF5PPWUBm6XcvAD7Mog2k%2526u%253DaHR0cHMlM2ElMmYlMmZ3d3cua2xvb2suY29tJTJmZW4tSU4lMmZkZXN0aW5hdGlvbiUyZmMxNjk0MjQtbmljZSUyZjEtdGhpbmdzLXRvLWRvJTJmJTNmX2N1cnJlbmN5JTNkSU5SJTI2dXRtX3NvdXJjZSUzZGJpbmclMjZ1dG1fb

In [7]:
hardcoded =[
    "https://bookchapter.org/kitaplar/Medical%20Diagnosis%20and%20Treatment%20Methods%20in%20Basic%20Medical%20Sciences.pdf",
    "https://applications.emro.who.int/dsaf/dsa236.pdf" 
]

In [8]:



def load_pdf(url):
    loader = PyPDFLoader(url)
    return loader.load()



In [14]:
vectorizer = SentenceTransformer("all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = []
for url in hardcoded:
    docs = load_pdf(url)
    documents.extend(docs)
chunks = text_splitter.split_documents(documents)
vectorstore = FAISS.from_documents(chunks, vectorizer)
retriever = vectorstore.as_retriever()
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4-0613", temperature=
    0.2),
    retriever=retriever,

    return_source_documents=True,
)
query = "What are the common diagnostic methods for diabetes?"

result = qa_chain(query)
print("Answer:", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")        

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5900.85it/s]


ImportError: pypdf package not found, please install it with `pip install pypdf`

In [9]:
def split_text(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    return text_splitter.split_documents(documents)

In [ ]:
def generate(query):
    result = qa_chain(query)
    print("Answer:", result["result"])
    print("\nSources:")
    for doc in result["source_documents"]:
        print(f"- {doc.metadata['source']}")
    return result

In [10]:
userquery = "What are the symptoms of diabetes?"


In [11]:
# Hugging Face collections are not directly loadable with datasets.load_dataset.
# Use a specific dataset repo ID or download files from huggingface_hub instead.


In [12]:
from datasets import load_dataset

dataset = load_dataset(
    "openlifescienceai/medmcqa",
    split="train"
)

print(dataset)
print(dataset[0])

Generating validation split: 100%|██████████| 4183/4183 [00:00<00:00, 184975.84 examples/s]

Dataset({
    features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
    num_rows: 182822
})
{'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': 2, 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}
